In [ ]:
import sys
import os

import cv2
import numpy as np
from paddleocr import PaddleOCR
from PyQt5.QtWidgets import (
    QApplication, QWidget, QLabel, QPushButton, QFileDialog,
    QHBoxLayout, QVBoxLayout, QMessageBox,
)
from PyQt5.QtGui import QPixmap, QImage
from PyQt5.QtCore import Qt


def Morph_Distinguish(img):
    """形态学方法从整幅图中提取车牌 ROI"""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (17, 17))
    tophat = cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, kernel)
    y = cv2.Sobel(tophat, cv2.CV_16S, 1, 0)
    absY = cv2.convertScaleAbs(y)
    _, binary = cv2.threshold(absY, 75, 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 15))
    Open = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (41, 15))
    close = cv2.morphologyEx(Open, cv2.MORPH_CLOSE, kernel)
    kernel_x = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 7))
    kernel_y = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 11))
    erode_y = cv2.morphologyEx(close, cv2.MORPH_ERODE, kernel_y)
    dilate_y = cv2.morphologyEx(erode_y, cv2.MORPH_DILATE, kernel_y)
    dilate_x = cv2.morphologyEx(dilate_y, cv2.MORPH_DILATE, kernel_x)
    erode_x = cv2.morphologyEx(dilate_x, cv2.MORPH_ERODE, kernel_x)
    kernel_e = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 9))
    erode = cv2.morphologyEx(erode_x, cv2.MORPH_ERODE, kernel_e)
    kernel_d = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 11))
    dilate = cv2.morphologyEx(erode, cv2.MORPH_DILATE, kernel_d)
    contours, _ = cv2.findContours(dilate, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for contour in contours:
        area = cv2.contourArea(contour)
        x, y, w, h = cv2.boundingRect(contour)
        if h * 2 < w < h * 8 and area > 500:
            ROI = img[y:y + h, x:x + w]
            return ROI
    return None


#AI-------------------------------Gemini 3 pro
# 问题：修改 Morph_Distinguish 函数，让它不仅返回车牌 ROI，还返回车牌在原图中的位置坐标 (x, y, w, h)，
# 这样可以在原图上画出红色框来标记车牌位置。
def Morph_Distinguish_with_box(img):
    """
    与 Morph_Distinguish 相同，但额外返回车牌在图像中的位置 (x, y, w, h)，
    方便在原图上画出红色框。
    """
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (17, 17))
    tophat = cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, kernel)
    y = cv2.Sobel(tophat, cv2.CV_16S, 1, 0)
    absY = cv2.convertScaleAbs(y)
    _, binary = cv2.threshold(absY, 75, 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 15))
    Open = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (41, 15))
    close = cv2.morphologyEx(Open, cv2.MORPH_CLOSE, kernel)
    kernel_x = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 7))
    kernel_y = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 11))
    erode_y = cv2.morphologyEx(close, cv2.MORPH_ERODE, kernel_y)
    dilate_y = cv2.morphologyEx(erode_y, cv2.MORPH_DILATE, kernel_y)
    dilate_x = cv2.morphologyEx(dilate_y, cv2.MORPH_DILATE, kernel_x)
    erode_x = cv2.morphologyEx(dilate_x, cv2.MORPH_ERODE, kernel_x)
    kernel_e = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 9))
    erode = cv2.morphologyEx(erode_x, cv2.MORPH_ERODE, kernel_e)
    kernel_d = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 11))
    dilate = cv2.morphologyEx(erode, cv2.MORPH_DILATE, kernel_d)
    contours, _ = cv2.findContours(dilate, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for contour in contours:
        area = cv2.contourArea(contour)
        x, y, w, h = cv2.boundingRect(contour)
        if h * 2 < w < h * 8 and area > 500:
            ROI = img[y:y + h, x:x + w]
            return ROI, (x, y, w, h)
    return None, None
#AI-------------------------------


def segment_characters(plate_bgr):
    """
    对提取出的车牌 ROI 进行灰度化、二值化和字符分割，
    返回：
    - binary：二值车牌图
    - strip：将每个字符横向拼接、用竖线分隔后的图像（白字黑底）
    """
    gray = cv2.cvtColor(plate_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)

    # OTSU 二值化
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 保证白字黑底
    if np.mean(binary) > 127:
        binary = 255 - binary

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)

    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    h, w = binary.shape
    char_imgs = []
#AI---------------------------Gemini 3 pro
# 问题：在字符分割函数中，添加过滤逻辑来去除噪声。过滤掉：
# 1. 高度小于车牌高度50%的字符（可能是噪声）
# 2. 宽度大于车牌宽度50%或小于5像素的字符（可能是粘连或噪声）
# 3. 面积小于车牌总面积1%的字符（小噪声点）
# 4. 宽高比小于0.2的极窄小块（类似"."的干扰）
    for cnt in contours:
        x, y, cw, ch = cv2.boundingRect(cnt)
        area = cw * ch

        # 过滤噪声
        if ch < 0.5 * h:
            continue
        if cw > 0.5 * w or cw < 5:
            continue
        if area < 0.01 * w * h:
            continue
        # 去掉类似 "." 的极窄小块
        if cw / float(ch) < 0.2:
            continue

        char_img = binary[y:y + ch, x:x + cw]
        char_imgs.append((x, char_img))
#AI-----------------------------
    if not char_imgs:
        return binary, None

    char_imgs.sort(key=lambda t: t[0])
    char_imgs = [img for _, img in char_imgs]

    target_h = 60
    sep_w = 4
    cells = []
    for ch_img in char_imgs:
        ch_h, ch_w = ch_img.shape
        scale = target_h / float(ch_h)
        new_w = int(ch_w * scale)
        resized = cv2.resize(ch_img, (new_w, target_h), interpolation=cv2.INTER_NEAREST)
        cells.append(resized)

    sep = np.ones((target_h, sep_w), dtype=np.uint8) * 255
    parts = []
    for i, cell in enumerate(cells):
        parts.append(cell)
        if i != len(cells) - 1:
            parts.append(sep)

    strip = np.concatenate(parts, axis=1)
    return binary, strip


#AI-------------------------------Chat GPT 4.0
# 问题：写一个函数来识别车牌的颜色（蓝色、黄色、绿色）。要求：
# 1. 识别的是车牌背景颜色，不是字体颜色
# 2. 通过分析车牌上下边缘区域的颜色来判断，避免字符区域的干扰
# 3. 使用HSV色彩空间进行颜色识别，更准确
# 4. 返回颜色名称（"蓝色"、"黄色"、"绿色"），如果识别不确定则返回"未知"
def recognize_plate_color(plate_bgr):
    """
    识别车牌背景颜色（不是字体颜色），返回颜色名称（黄色、蓝色、绿色）
    通过统计车牌上下边缘区域的颜色来确定背景色，避免字符区域的干扰
    """
    # 转换到HSV色彩空间，更利于颜色识别
    hsv = cv2.cvtColor(plate_bgr, cv2.COLOR_BGR2HSV)
    
    h, w = hsv.shape[:2]
    
    # 取车牌上下边缘区域（通常没有字符，只有背景色）
    # 上下各取15%的区域
    edge_height = int(h * 0.15)
    top_region = hsv[0:edge_height, :]
    bottom_region = hsv[h-edge_height:h, :]
    
    # 合并上下边缘区域
    background_region = np.vstack([top_region, bottom_region])
    total_pixels = background_region.shape[0] * background_region.shape[1]
    
    # 定义颜色范围（HSV）
    # 蓝色（中国车牌常见）
    blue_mask = cv2.inRange(background_region, np.array([100, 43, 46]), np.array([124, 255, 255]))
    blue_count = np.sum(blue_mask > 0)
    
    # 黄色（中国车牌常见）
    yellow_mask = cv2.inRange(background_region, np.array([11, 43, 46]), np.array([34, 255, 255]))
    yellow_count = np.sum(yellow_mask > 0)
    
    # 绿色（新能源车牌）
    green_mask = cv2.inRange(background_region, np.array([35, 43, 46]), np.array([77, 255, 255]))
    green_count = np.sum(green_mask > 0)
    
    # 计算各颜色占比
    color_ratios = {
        '蓝色': blue_count / total_pixels,
        '黄色': yellow_count / total_pixels,
        '绿色': green_count / total_pixels
    }
    
    # 返回占比最大的颜色
    color = max(color_ratios, key=color_ratios.get)
    
    # 如果占比太低，可能是识别错误，返回未知
    if color_ratios[color] < 0.15:
        return "未知"
    
    return color
#AI-------------------------------

class PlateWindow(QWidget):
    def __init__(self):
        super().__init__()
        os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
        self.ocr = PaddleOCR(use_angle_cls=True, lang="ch")
        self.img_bgr = None
        self.roi_bgr = None
        self.init_ui()

    def init_ui(self):
        self.setWindowTitle("车牌识别演示")

        bg_path = r"猫猫.jpg"
        bg_path_url = bg_path.replace("\\", "/")
        self.setStyleSheet(
            f"""
            QWidget {{
                background-image: url('{bg_path_url}');
                background-position: center;
                background-repeat: no-repeat;
            }}
            QPushButton {{
                background-color: rgba(0, 0, 0, 180);
                color: #ffffff;  /* 白色文字 */
                font-weight: 700;  /* 更粗一些 */
                border: 1px solid #cccccc;
                padding: 4px 8px;
            }}
            """
        )

        # 左侧：原图显示
        self.orig_label = QLabel("原图区域")
        self.orig_label.setAlignment(Qt.AlignCenter)
        self.orig_label.setStyleSheet("background-color: #222; color: #ffffff; font-weight: 700;")
        self.orig_label.setMinimumSize(400, 300)

        # 右侧：车牌提取结果（标题 + 图片）
        self.plate_title = QLabel("车牌提取结果：")
        self.plate_title.setStyleSheet("font-size: 14px; color: #ffffff; font-weight: 700;")
        self.plate_label = QLabel()
        self.plate_label.setAlignment(Qt.AlignCenter)
        self.plate_label.setStyleSheet("background-color: #222; color: #ffffff; font-weight: 700;")
        self.plate_label.setMinimumSize(300, 80)

        # 右侧：车牌颜色识别结果
        self.color_title = QLabel("车牌颜色：")
        self.color_title.setStyleSheet("font-size: 14px; color: #ffffff; font-weight: 700;")
        self.color_label = QLabel("")
        self.color_label.setAlignment(Qt.AlignLeft | Qt.AlignVCenter)
        self.color_label.setStyleSheet("font-size: 16px; color: #ffffff; font-weight: 700;")

        # 右侧：字符分割结果（标题 + 条带）
        self.seg_title = QLabel("字符分割结果：")
        self.seg_title.setStyleSheet("font-size: 14px; color: #ffffff; font-weight: 700;")
        self.seg_label = QLabel()
        self.seg_label.setAlignment(Qt.AlignCenter)
        self.seg_label.setStyleSheet("background-color: #222; color: #ffffff; font-weight: 700;")
        self.seg_label.setMinimumSize(300, 80)

        # 右侧：车牌识别文字结果
        self.result_title = QLabel("车牌识别结果：")
        self.result_title.setStyleSheet("font-size: 14px; color: #ffffff; font-weight: 700;")
        self.result_label = QLabel("")
        self.result_label.setAlignment(Qt.AlignLeft | Qt.AlignVCenter)
        self.result_label.setStyleSheet("font-size: 16px; color: #ffffff; font-weight: 700;")

        # 底部按钮栏
        self.btn_show = QPushButton("显示图片")
        self.btn_plate = QPushButton("车牌提取")
        self.btn_seg = QPushButton("字符分割")
        self.btn_ocr = QPushButton("车牌识别")

        self.btn_show.clicked.connect(self.choose_image)
        self.btn_plate.clicked.connect(self.extract_plate)
        self.btn_seg.clicked.connect(self.segment_plate_chars)
        self.btn_ocr.clicked.connect(self.recognize_plate)

        left_layout = QVBoxLayout()
        left_layout.addWidget(self.orig_label)

        btn_layout = QHBoxLayout()
        btn_layout.addWidget(self.btn_show)
        btn_layout.addWidget(self.btn_plate)
        btn_layout.addWidget(self.btn_seg)
        btn_layout.addWidget(self.btn_ocr)
        left_layout.addLayout(btn_layout)

        right_layout = QVBoxLayout()
        right_layout.addWidget(self.plate_title)
        right_layout.addWidget(self.plate_label)
        right_layout.addWidget(self.color_title)
        right_layout.addWidget(self.color_label)
        right_layout.addWidget(self.seg_title)
        right_layout.addWidget(self.seg_label)
        right_layout.addWidget(self.result_title)
        right_layout.addWidget(self.result_label)

        main_layout = QHBoxLayout()
        main_layout.addLayout(left_layout, 2)
        main_layout.addLayout(right_layout, 1)

        self.setLayout(main_layout)
        self.resize(1000, 600)

    # -------- 按钮功能 --------

    def choose_image(self):
        fname, _ = QFileDialog.getOpenFileName(
            self, "选择车牌图片", "", "Images (*.jpg *.jpeg *.png *.bmp)"
        )
        if not fname:
            return

        self.img_bgr = cv2.imdecode(np.fromfile(fname, dtype=np.uint8), cv2.IMREAD_COLOR)
        if self.img_bgr is None:
            QMessageBox.warning(self, "错误", "无法读取图片")
            return

        # 新图时清空右侧结果
        self.roi_bgr = None
        self.plate_label.clear()
        self.color_label.setText("")
        self.seg_label.clear()
        self.result_label.setText("")

        self.show_image_on_label(self.img_bgr, self.orig_label)

    def extract_plate(self):
        if self.img_bgr is None:
            QMessageBox.information(self, "提示", "请先点击“显示图片”选择一张图片")
            return

        # 1) 生成用于检测的图像（做放大 + 遮挡左下角文字）
        img_resized = cv2.resize(
            self.img_bgr,
            (int(self.img_bgr.shape[1] * 1.5), int(self.img_bgr.shape[0] * 1.5)),
        )
        img_for_detect = img_resized.copy()
        h, w, _ = img_for_detect.shape
        img_for_detect[int(h * 0.7):h, 0:int(w * 0.4)] = 0
        img_rgb = cv2.cvtColor(img_for_detect, cv2.COLOR_BGR2RGB)

        roi, box = Morph_Distinguish_with_box(img_rgb)
        if roi is None or box is None:
            QMessageBox.information(self, "提示", "没有检测到车牌")
            self.roi_bgr = None
            self.plate_label.setText("未检测到车牌")
            self.color_label.setText("")
            return

        # 放大 ROI 用于后续显示和识别
        roi = cv2.resize(roi, None, fx=3.0, fy=3.0, interpolation=cv2.INTER_CUBIC)
        self.roi_bgr = cv2.cvtColor(roi, cv2.COLOR_RGB2BGR)
        self.show_image_on_label(self.roi_bgr, self.plate_label)

        # 识别车牌颜色
        plate_color = recognize_plate_color(self.roi_bgr)
        self.color_label.setText(plate_color)

        # 2) 在“未遮挡”的放大原图上画出车牌红框并显示在左侧，不再出现大黑块
        x, y, w, h = box
        boxed_bgr = img_resized.copy()
        cv2.rectangle(boxed_bgr, (x, y), (x + w, y + h), (0, 0, 255), 3)
        self.show_image_on_label(boxed_bgr, self.orig_label)

    def segment_plate_chars(self):
        """字符分割：对已提取的车牌做字符分割"""
        if self.roi_bgr is None:
            QMessageBox.information(self, "提示", "请先点击“车牌提取”")
            return

        _, strip = segment_characters(self.roi_bgr)
        if strip is not None:
            self.show_gray_on_label(strip, self.seg_label)
        else:
            self.seg_label.setText("未能分割字符")

    def recognize_plate(self):
        """车牌识别：对已提取的车牌做 OCR"""
        if self.roi_bgr is None:
            QMessageBox.information(self, "提示", "请先点击“车牌提取”")
            return

        result = self.ocr.ocr(self.roi_bgr, cls=True)
        plate_text_clean = "未识别到"
        if result and len(result) > 0 and result[0]:
            plate_text = ""
            for line in result[0]:
                if line and len(line) >= 2:
                    text = line[1][0]
                    plate_text += text
            
            plate_text_clean = plate_text.replace(" ", "").replace(".", "")
            
        self.result_label.setText(plate_text_clean)

    def show_image_on_label(self, bgr_img, label):
        rgb = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
        h, w, ch = rgb.shape
        bytes_per_line = ch * w
        qimg = QImage(rgb.data, w, h, bytes_per_line, QImage.Format_RGB888)
        pix = QPixmap.fromImage(qimg)
        pix = pix.scaled(
            label.width(), label.height(), Qt.KeepAspectRatio, Qt.SmoothTransformation
        )
        label.setPixmap(pix)

    def show_gray_on_label(self, gray_img, label):
        h, w = gray_img.shape
        qimg = QImage(gray_img.data, w, h, w, QImage.Format_Grayscale8)
        pix = QPixmap.fromImage(qimg)
        pix = pix.scaled(
            label.width(), label.height(), Qt.KeepAspectRatio, Qt.SmoothTransformation
        )
        label.setPixmap(pix)


if __name__ == "__main__":
    app = QApplication(sys.argv)
    win = PlateWindow()
    win.show()
    sys.exit(app.exec_())


[2025/12/20 21:14:28] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, use_gcu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\Lenovo/.paddleocr/whl\\det\\ch\\ch_PP-OCRv4_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\Lenovo/.paddleocr/whl\\rec\\ch\\ch_PP-OCRv4_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320